In [ ]:
!pip install -q \
transformers \
accelerate \
openpyxl \
scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [ ]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

In [ ]:
print(df.columns)

Index(['id', 'comment', 'label'], dtype='object')


In [ ]:
df = df[['comment', 'label']]

In [ ]:
df.dropna(inplace=True)

In [ ]:
df['label'] = df['label'].astype(int)

In [ ]:
print(df.head())

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0
label
0    1577
1    1471
Name: count, dtype: int64


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "keshan/SinhalaBERTo"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "keshan/SinhalaBERTo",
    num_labels=2
)

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/721k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/334M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: keshan/SinhalaBERTo
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [ ]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

In [ ]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=10,

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    warmup_ratio=0.1,

    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    save_total_limit=1
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.595562,0.610714,0.649180,0.645980,0.649963,0.646592
2,0.578939,0.608143,0.709836,0.709817,0.710272,0.710465
3,0.303196,0.812006,0.665574,0.656338,0.675233,0.660521
4,0.276766,1.180730,0.649180,0.639886,0.678411,0.655828
5,0.169452,1.642838,0.691803,0.691325,0.696460,0.694125
6,0.100711,2.107492,0.667213,0.662105,0.670957,0.663642
7,0.000284,2.295204,0.686885,0.686642,0.686603,0.686773
8,0.001117,2.449719,0.670492,0.670470,0.671911,0.671661
9,0.012472,2.518569,0.675410,0.675284,0.675353,0.675579
10,0.000210,2.556448,0.678689,0.678602,0.678750,0.678980


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3050, training_loss=0.23441194739608767, metrics={'train_runtime': 557.6254, 'train_samples_per_second': 43.721, 'train_steps_per_second': 5.47, 'total_flos': 807388794808320.0, 'train_loss': 0.23441194739608767, 'epoch': 10.0})

In [ ]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.000210,0.608143,10,0.709836,0.709817,0.710272,0.710465


{'eval_loss': 0.6081429123878479, 'eval_accuracy': 0.7098360655737705, 'eval_f1': 0.7098165692400725, 'eval_precision': 0.7102721827312908, 'eval_recall': 0.7104645655730647}


In [ ]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(-1)

cm = confusion_matrix(test_labels, preds)

print(cm)

[[219  97]
 [ 80 214]]


In [ ]:
print(
    classification_report(
        test_labels,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.73      0.69      0.71       316
           1       0.69      0.73      0.71       294

    accuracy                           0.71       610
   macro avg       0.71      0.71      0.71       610
weighted avg       0.71      0.71      0.71       610



In [ ]:
model.save_pretrained(
    "./sarcasm_model"
)

tokenizer.save_pretrained(
    "./sarcasm_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sarcasm_model/tokenizer_config.json', './sarcasm_model/tokenizer.json')

In [ ]:
!zip -r sarcasm_model.zip sarcasm_model

  adding: sarcasm_model/ (stored 0%)
  adding: sarcasm_model/tokenizer_config.json (deflated 50%)
  adding: sarcasm_model/model.safetensors (deflated 7%)
  adding: sarcasm_model/tokenizer.json (deflated 83%)
  adding: sarcasm_model/config.json (deflated 50%)


In [ ]:
from google.colab import files

files.download("sarcasm_model.zip")

FileNotFoundError: Cannot find file: sarcasm_model.zip

In [ ]:
text = "අද නම් හරිම ලස්සන වැඩක් කරලා 😂"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

if prediction.item() == 1:
    print("Sarcastic")
else:
    print("Non-Sarcastic")

Prediction: 0
Non-Sarcastic
